# 도메인 분류 검증 (Domain Evaluation)

**목적**: ContentAnalyzer의 도메인 분류 정확도를 정량적으로 평가하고, 개선 사이클을 반복할 수 있는 기반을 만든다.

### 워크플로우
1. 층화 샘플링으로 평가 데이터셋 추출 (100~150건)
2. LLM 보조 라벨링 → 수동 검수로 gold standard 생성
3. ContentAnalyzer 분류 결과 vs gold standard 비교
4. 정확도/재현율/F1 계산 → 키워드 보완 → 재측정

## 1. 데이터 로드 및 샘플링

In [ ]:
import pandas as pd
import numpy as np
import re
import sys
from collections import Counter

from utils.db import load_job_postings

sys.path.insert(0, '../../')
from app.pipeline.normalizers.content import ContentAnalyzer

# 전체 데이터 로드
df = load_job_postings()
print(f"전체 공고 수: {len(df)}")

In [ ]:
# ContentAnalyzer로 전체 데이터 분류
analyzer = ContentAnalyzer()

def combine_body(row):
    parts = []
    for col in ['position_detail', 'main_tasks', 'requirements', 'preferred']:
        text = row.get(col)
        if isinstance(text, str) and text.strip():
            parts.append(text)
    return ' '.join(parts)

df['body'] = df.apply(combine_body, axis=1)
df['analyzer_result'] = df['body'].apply(analyzer.analyze)
df['predicted_domain'] = df['analyzer_result'].apply(lambda r: r.domain)
df['predicted_confidence'] = df['analyzer_result'].apply(lambda r: r.domain_confidence)

# 현재 분류 현황
print("=== ContentAnalyzer 분류 현황 ===")
print(f"분류됨: {df['predicted_domain'].notna().sum()}건")
print(f"미분류: {df['predicted_domain'].isna().sum()}건")
print(f"\n도메인별 분포:")
print(df['predicted_domain'].value_counts().to_string())

In [ ]:
# position_detail 컬럼 상태 확인 (문제 1: 내용 부족 케이스)
df['pd_length'] = df['position_detail'].fillna('').str.len()

print("=== position_detail 길이 분포 ===")
print(f"비어있음 (0자):    {(df['pd_length'] == 0).sum()}건")
print(f"매우 짧음 (<50자): {((df['pd_length'] > 0) & (df['pd_length'] < 50)).sum()}건")
print(f"짧음 (50~200자):   {((df['pd_length'] >= 50) & (df['pd_length'] < 200)).sum()}건")
print(f"보통 (200~500자):  {((df['pd_length'] >= 200) & (df['pd_length'] < 500)).sum()}건")
print(f"충분 (500자+):     {(df['pd_length'] >= 500).sum()}건")

print(f"\n통계:")
print(df['pd_length'].describe())

In [ ]:
# 층화 샘플링: 도메인 분류 결과 + position_detail 길이 기준
# - 분류된 공고: 각 도메인에서 균등 추출
# - 미분류 공고: 별도 추출 (이 그룹에서 놓치는 도메인이 있는지 확인)
# - position_detail 짧은 공고: 별도 추출 (문제 1 검증)

np.random.seed(42)
sample_parts = []

# (A) 분류된 공고 - 각 도메인에서 5건씩
classified = df[df['predicted_domain'].notna()]
for domain, group in classified.groupby('predicted_domain'):
    n = min(5, len(group))
    sample_parts.append(group.sample(n=n, random_state=42))

# (B) 미분류 공고 - 50건
unclassified = df[df['predicted_domain'].isna()]
n_unclassified = min(50, len(unclassified))
sample_parts.append(unclassified.sample(n=n_unclassified, random_state=42))

# (C) position_detail 짧은 공고 - 20건 (위에서 이미 뽑힌 것 제외)
short_pd = df[df['pd_length'] < 50]
already_sampled_ids = pd.concat(sample_parts)['id'].tolist()
short_pd_remaining = short_pd[~short_pd['id'].isin(already_sampled_ids)]
n_short = min(20, len(short_pd_remaining))
if n_short > 0:
    sample_parts.append(short_pd_remaining.sample(n=n_short, random_state=42))

# 합치기
eval_sample = pd.concat(sample_parts).drop_duplicates(subset='id').reset_index(drop=True)

print(f"평가 샘플 수: {len(eval_sample)}건")
print(f"\n샘플 내 predicted_domain 분포:")
print(eval_sample['predicted_domain'].value_counts(dropna=False).to_string())

## 2. Gold Standard 라벨링

**방법**: LLM 보조 라벨링 후 수동 검수

도메인 후보:  
`금융/핀테크`, `바이오/헬스케어`, `커머스/리테일`, `게임`, `콘텐츠/미디어`,  
`SaaS/B2B`, `교육/에듀테크`, `모빌리티/자동차`, `보안/사이버보안`, `AI/딥테크`,  
`기타` (위 10개에 해당하지 않는 경우)

In [ ]:
# 라벨링용 데이터 준비
# LLM에 전달할 입력: position + position_detail (+ 필요시 main_tasks)
labeling_df = eval_sample[[
    'id', 'posting_id', 'position', 'position_detail', 
    'main_tasks', 'requirements', 'predicted_domain'
]].copy()

# gold standard 컬럼 추가 (빈 값 - 라벨링 후 채움)
labeling_df['gold_domain'] = None          # 정답 도메인
labeling_df['gold_confidence'] = None       # 라벨러 확신도 (high/medium/low)
labeling_df['labeling_note'] = None         # 판단 근거 메모

print(f"라벨링 대상: {len(labeling_df)}건")
labeling_df[['posting_id', 'position', 'predicted_domain']].head(20)

In [ ]:
# TODO: LLM 보조 라벨링
# 아래 두 가지 방법 중 선택:
#
# 방법 A) OpenAI API / Claude API 직접 호출
#   - 각 공고의 body 텍스트를 LLM에 전달
#   - 도메인 + 근거 키워드를 JSON으로 반환받기
#
# 방법 B) CSV로 내보내고 외부에서 라벨링
#   - labeling_df.to_csv('../outputs/eval_labeling.csv', index=False)
#   - 수동 또는 외부 도구로 gold_domain 컬럼 채우기
#   - labeled = pd.read_csv('../outputs/eval_labeling_done.csv')

# 우선 CSV 내보내기
labeling_df.to_csv('../outputs/eval_labeling.csv', index=False)
print("../outputs/eval_labeling.csv 저장 완료")

In [ ]:
# 라벨링 완료 후 gold standard 로드
# gold_df = pd.read_csv('../outputs/eval_labeling_done.csv')
# print(f"라벨링 완료: {gold_df['gold_domain'].notna().sum()} / {len(gold_df)}건")
# print(f"\ngold_domain 분포:")
# print(gold_df['gold_domain'].value_counts(dropna=False).to_string())

## 3. ContentAnalyzer 평가

Gold standard vs ContentAnalyzer 예측 비교  
→ 정확도, 도메인별 Precision/Recall/F1, 혼동행렬

In [ ]:
# TODO: gold standard 로드 후 실행
#
# from sklearn.metrics import classification_report, confusion_matrix
# import seaborn as sns
# import matplotlib.pyplot as plt
# from matplotlib import rc
# rc('font', family='AppleGothic')
# plt.rcParams['axes.unicode_minus'] = False
#
# # 미분류를 '기타'로 통일
# gold_df['gold_filled'] = gold_df['gold_domain'].fillna('기타')
# gold_df['pred_filled'] = gold_df['predicted_domain'].fillna('기타')
#
# # Classification Report
# labels = sorted(set(gold_df['gold_filled'].unique()) | set(gold_df['pred_filled'].unique()))
# print(classification_report(gold_df['gold_filled'], gold_df['pred_filled'], labels=labels, zero_division=0))
#
# # 혼동행렬
# cm = confusion_matrix(gold_df['gold_filled'], gold_df['pred_filled'], labels=labels)
# fig, ax = plt.subplots(figsize=(12, 10))
# sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax)
# ax.set_xlabel('Predicted')
# ax.set_ylabel('Gold Standard')
# ax.set_title('ContentAnalyzer 도메인 분류 혼동행렬')
# plt.tight_layout()
# plt.show()

In [ ]:
# TODO: 오분류 분석
#
# # 틀린 케이스만 추출
# errors = gold_df[gold_df['gold_filled'] != gold_df['pred_filled']].copy()
# print(f"오분류: {len(errors)}건 / {len(gold_df)}건 ({len(errors)/len(gold_df)*100:.1f}%)")
#
# # 오분류 유형별 집계
# error_types = errors.groupby(['gold_filled', 'pred_filled']).size().sort_values(ascending=False)
# print("\n오분류 유형 (gold → predicted):")
# print(error_types.to_string())
#
# # 미분류(False Negative): gold에는 도메인이 있지만 analyzer가 못 찾은 경우
# fn = errors[errors['pred_filled'] == '기타']
# print(f"\n미분류 False Negative: {len(fn)}건")
# if len(fn) > 0:
#     for _, row in fn.head(10).iterrows():
#         print(f"  [{row['gold_filled']}] {row['position']}")

## 4. 개선 및 재평가

오분류 분석 결과를 바탕으로 ContentAnalyzer 키워드 보강 → 재평가  
이 사이클을 반복하여 정확도를 높인다.

In [ ]:
# TODO: 오분류 케이스에서 누락된 키워드 후보 추출
#
# from sklearn.feature_extraction.text import TfidfVectorizer
#
# TOKEN_PATTERN = r'[가-힣]{2,}|[A-Za-z][A-Za-z0-9/\.\-]{1,}'
#
# # False Negative (미분류) 문서들의 키워드 추출
# fn_corpus = fn['body'].tolist()
# if fn_corpus:
#     tfidf = TfidfVectorizer(
#         token_pattern=TOKEN_PATTERN, ngram_range=(1, 2),
#         max_features=3000, min_df=1, max_df=0.9
#     )
#     X = tfidf.fit_transform(fn_corpus)
#     features = tfidf.get_feature_names_out()
#     scores = X.sum(axis=0).A1
#     top_kws = sorted(zip(features, scores), key=lambda x: -x[1])[:30]
#     print("미분류 공고에서 빈출 키워드 (ContentAnalyzer에 추가 후보):")
#     for kw, s in top_kws:
#         print(f"  {s:.3f}  {kw}")

In [ ]:
# TODO: 키워드 보강 후 재평가
# ContentAnalyzer 수정 후 이 셀 재실행으로 개선 효과 즉시 확인
#
# import importlib
# import backend.pipeline.normalizers.content as content_mod
# importlib.reload(content_mod)
# analyzer_v2 = content_mod.ContentAnalyzer()
#
# gold_df['pred_v2'] = gold_df['body'].apply(lambda t: analyzer_v2.analyze(t).domain)
# gold_df['pred_v2_filled'] = gold_df['pred_v2'].fillna('기타')
#
# print("=== v2 평가 결과 ===")
# print(classification_report(gold_df['gold_filled'], gold_df['pred_v2_filled'], labels=labels, zero_division=0))